In [ ]:
import pandas as pd

df = pd.read_csv('train.csv.zip', index_col=0)
print(df.shape, df['SeriousDlqin2yrs'].mean())
print(df.isna().mean()[lambda s: s > 0])

(104805, 11) 0.066313630074901
MonthlyIncome         0.198283
NumberOfDependents    0.026230
dtype: float64


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

print(df['age'].min())
late = ['NumberOfTime30-59DaysPastDueNotWorse','NumberOfTime60-89DaysPastDueNotWorse','NumberOfTimes90DaysLate']
print(df[late].max())

df = df[df['age'] >= 18].copy()
for c in late:
    df[c] = df[c].where(df[c] < 90)        # 96/98 are placeholder codes, not real counts
df['income_missing'] = df['MonthlyIncome'].isna().astype(int)

X, y = df.drop(columns='SeriousDlqin2yrs'), df['SeriousDlqin2yrs']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

med = X_tr.median()                         # fit on train only, apply to both
X_tr, X_te = X_tr.fillna(med), X_te.fillna(med)

21
NumberOfTime30-59DaysPastDueNotWorse    13.0
NumberOfTime60-89DaysPastDueNotWorse    11.0
NumberOfTimes90DaysLate                 17.0
dtype: float64


In [ ]:
cap_cols = ['RevolvingUtilizationOfUnsecuredLines','DebtRatio','MonthlyIncome']
print(X_tr[cap_cols].quantile([.5,.9,.99,.999]))

caps = X_tr[cap_cols].quantile(.99)          # from train only
X_tr[cap_cols] = X_tr[cap_cols].clip(upper=caps, axis='columns')
X_te[cap_cols] = X_te[cap_cols].clip(upper=caps, axis='columns')

def woe_iv(x, y, bins=10):
    b = pd.qcut(x, bins, duplicates='drop')
    t = pd.crosstab(b, y) + 0.5               # smoothing avoids log(0)
    t.columns = ['good', 'bad']
    t = t / t.sum()
    t['woe'] = np.log(t['good'] / t['bad'])
    t['iv'] = (t['good'] - t['bad']) * t['woe']
    return t, t['iv'].sum()

feats = [c for c in X_tr.columns if c != 'income_missing']
iv = pd.Series({c: woe_iv(X_tr[c], y_tr)[1] for c in feats}).sort_values(ascending=False)
print(iv)

       RevolvingUtilizationOfUnsecuredLines    DebtRatio  MonthlyIncome
0.500                              0.156051     0.368262         5400.0
0.900                              0.980044  1290.000000        10800.0
0.990                              1.090006  4983.000000        23250.0
0.999                              1.090022  4983.000000        23250.0
RevolvingUtilizationOfUnsecuredLines    1.099936
NumberOfTime30-59DaysPastDueNotWorse    0.407475
age                                     0.277282
MonthlyIncome                           0.074049
DebtRatio                               0.071470
NumberOfOpenCreditLinesAndLoans         0.062974
NumberOfDependents                      0.025368
NumberRealEstateLoansOrLines            0.013472
NumberOfTimes90DaysLate                 0.000000
NumberOfTime60-89DaysPastDueNotWorse    0.000000
dtype: float64


In [ ]:
def woe_tab(b, y):
    t = pd.crosstab(b, y) + 0.5
    t.columns = ['good', 'bad']
    t = t / t.sum()
    t['woe'] = np.log(t['good'] / t['bad'])
    t['iv'] = (t['good'] - t['bad']) * t['woe']
    return t

count_cols = ['NumberOfTime30-59DaysPastDueNotWorse',
              'NumberOfTime60-89DaysPastDueNotWorse',
              'NumberOfTimes90DaysLate']
for c in count_cols:
    t = woe_tab(X_tr[c].clip(upper=3), y_tr)
    print(c, 'IV =', round(t['iv'].sum(), 3))
    print(t['woe'].round(2).to_dict())

for c in ['RevolvingUtilizationOfUnsecuredLines', 'age', 'MonthlyIncome']:
    t = woe_tab(pd.qcut(X_tr[c], 10, duplicates='drop'), y_tr)
    print(c); print(t['woe'].round(2))

NumberOfTime30-59DaysPastDueNotWorse IV = 0.672
{0.0: 0.5, 1.0: -0.9, 2.0: -1.59, 3.0: -2.16}
NumberOfTime60-89DaysPastDueNotWorse IV = 0.525
{0.0: 0.26, 1.0: -1.79, 2.0: -2.56, 3.0: -3.1}
NumberOfTimes90DaysLate IV = 0.818
{0.0: 0.36, 1.0: -1.94, 2.0: -2.64, 3.0: -3.12}
RevolvingUtilizationOfUnsecuredLines
RevolvingUtilizationOfUnsecuredLines
(-0.001, 0.00319]    1.05
(0.00319, 0.0195]    1.75
(0.0195, 0.0437]     1.58
(0.0437, 0.0838]     1.25
(0.0838, 0.156]      1.12
(0.156, 0.273]       0.59
(0.273, 0.447]       0.18
(0.447, 0.698]      -0.28
(0.698, 0.98]       -0.97
(0.98, 1.09]        -1.45
Name: woe, dtype: float64
age
age
(20.999, 33.0]   -0.63
(33.0, 39.0]     -0.41
(39.0, 44.0]     -0.26
(44.0, 48.0]     -0.19
(48.0, 52.0]     -0.15
(52.0, 56.0]      0.00
(56.0, 61.0]      0.32
(61.0, 65.0]      0.59
(65.0, 72.0]      1.01
(72.0, 105.0]     1.24
Name: woe, dtype: float64
MonthlyIncome
MonthlyIncome
(-0.001, 2313.0]     -0.29
(2313.0, 3400.0]     -0.39
(3400.0, 4337.0]     -

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

U = [-np.inf,0.0437,0.0838,0.156,0.273,0.447,0.698,0.98,np.inf]
A = [-np.inf,33,39,44,48,52,56,61,65,72,np.inf]
ok = X_tr['income_missing'] == 0
I = np.unique(np.r_[-np.inf, X_tr.loc[ok,'MonthlyIncome'].quantile(np.linspace(.125,.875,7)).values, np.inf])
cnt = ['NumberOfTime30-59DaysPastDueNotWorse','NumberOfTime60-89DaysPastDueNotWorse','NumberOfTimes90DaysLate']

def binned(X):
    B = pd.DataFrame(index=X.index)
    B['util'] = pd.cut(X['RevolvingUtilizationOfUnsecuredLines'], U).astype(str)
    B['age'] = pd.cut(X['age'], A).astype(str)
    B['inc'] = pd.cut(X['MonthlyIncome'], I).astype(str).where(X['income_missing']==0, 'missing')
    for c in cnt: B[c] = X[c].clip(upper=3).round().astype(int).astype(str)
    return B

B_tr, B_te = binned(X_tr), binned(X_te)
maps = {c: woe_tab(B_tr[c], y_tr)['woe'] for c in B_tr}
W_tr = B_tr.apply(lambda s: s.map(maps[s.name])).fillna(0)
W_te = B_te.apply(lambda s: s.map(maps[s.name])).fillna(0)
for c in ['util','inc']: print(c, maps[c].round(2).to_dict())

m = LogisticRegression(max_iter=1000).fit(W_tr, y_tr)
p = m.predict_proba(W_te)[:, 1]
auc = roc_auc_score(y_te, p); fpr, tpr, _ = roc_curve(y_te, p)
print(dict(zip(W_tr.columns, m.coef_[0].round(2))))
print('AUC', round(auc,3), 'Gini', round(2*auc-1,3), 'KS', round((tpr-fpr).max(),3))

util {'(-inf, 0.0437]': 1.42, '(0.0437, 0.0838]': 1.26, '(0.0838, 0.156]': 1.11, '(0.156, 0.273]': 0.59, '(0.273, 0.447]': 0.18, '(0.447, 0.698]': -0.28, '(0.698, 0.98]': -0.97, '(0.98, inf]': -1.45}
inc {'(-inf, 2313.0]': -0.29, '(10800.0, inf]': 0.53, '(2313.0, 3400.0]': -0.39, '(3400.0, 4339.5]': -0.26, '(4339.5, 5400.0]': -0.14, '(5400.0, 6630.0]': 0.01, '(6630.0, 8284.0]': 0.26, '(8284.0, 10800.0]': 0.32, 'missing': 0.16}
{'util': np.float64(-0.65), 'age': np.float64(-0.47), 'inc': np.float64(-0.35), 'NumberOfTime30-59DaysPastDueNotWorse': np.float64(-0.55), 'NumberOfTime60-89DaysPastDueNotWorse': np.float64(-0.4), 'NumberOfTimes90DaysLate': np.float64(-0.53)}
AUC 0.853 Gini 0.707 KS 0.557


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(300, min_samples_leaf=50, n_jobs=-1, random_state=42).fit(X_tr, y_tr)
print('RF AUC', round(roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1]), 3))

factor = 20/np.log(2); offset = 600 - factor*np.log(50)   # 600 pts at 50:1 good:bad, 20 pts to double odds
s_tr = offset - factor*m.decision_function(W_tr)
s_te = offset - factor*m.decision_function(W_te)

edges = np.unique(np.quantile(s_tr, np.linspace(0, 1, 11))); edges[0], edges[-1] = -np.inf, np.inf
a = pd.Series(pd.cut(s_tr, edges)).value_counts(normalize=True, sort=False)
b = pd.Series(pd.cut(s_te, edges)).value_counts(normalize=True, sort=False)
print('PSI', round(((b-a)*np.log(b/a)).sum(), 4))

te = pd.DataFrame({'score': s_te, 'bad': y_te.values})
for q in [0, .1, .2, .3, .4, .5]:          # reject the lowest-scoring q share
    cut = te['score'].quantile(q); ap = te[te['score'] >= cut]
    print(f'reject {q:.0%}: cutoff {cut:.0f}, approval {len(ap)/len(te):.0%}, bad rate {ap.bad.mean():.2%}')

RF AUC 0.866
PSI 0.0009
reject 0%: cutoff 411, approval 100%, bad rate 6.63%
reject 10%: cutoff 540, approval 90%, bad rate 3.48%
reject 20%: cutoff 557, approval 80%, bad rate 2.53%
reject 30%: cutoff 571, approval 70%, bad rate 1.76%
reject 40%: cutoff 582, approval 60%, bad rate 1.28%
reject 50%: cutoff 592, approval 50%, bad rate 1.11%


In [ ]:
co = dict(zip(W_tr.columns, m.coef_[0])); n = len(co)
card = pd.DataFrame([(c, str(b), round(w, 3), round(offset/n - factor*(co[c]*w + m.intercept_[0]/n)))
                     for c in co for b, w in maps[c].items()],
                    columns=['variable', 'bin', 'woe', 'points'])
card.to_csv('scorecard_points.csv', index=False)

pm = {c: card[card.variable == c].set_index('bin')['points'] for c in co}
chk = B_te.apply(lambda s: s.map(pm[s.name])).sum(axis=1)
print('max diff vs model score:', round((chk - s_te).abs().max(), 2))    # rounding only, expect under ~3

te['band'] = pd.qcut(te['score'], 10, duplicates='drop').astype(str)
bands = te.groupby('band', observed=True).agg(applicants=('bad', 'size'), bads=('bad', 'sum'),
                                              bad_rate=('bad', 'mean'), avg_score=('score', 'mean')).reset_index()
bands.to_csv('score_bands.csv', index=False)

rows = []
for q in np.arange(0, .55, .05):
    cut = te['score'].quantile(q); ap = te[te['score'] >= cut]
    rows.append((q, cut, len(ap)/len(te), ap.bad.mean()))
pd.DataFrame(rows, columns=['reject_share', 'cutoff', 'approval_rate', 'bad_rate']).to_csv('cutoff_table.csv', index=False)
print(bands)

max diff vs model score: 1.91
                            band  applicants  bads  bad_rate   avg_score
0  (411.27000000000004, 539.859]        3167  1107  0.349542  508.072519
1             (539.859, 557.464]        3128   346  0.110614  549.430614
2             (557.464, 571.165]        3162   247  0.078115  564.895175
3             (571.165, 582.402]        3145   143  0.045469  577.090014
4             (582.402, 592.281]        3130    67  0.021406  587.417650
5             (592.281, 599.211]        3137    59  0.018808  595.915188
6             (599.211, 604.166]        3164    40  0.012642  601.602525
7             (604.166, 609.833]        3171    30  0.009461  606.825229
8             (609.833, 618.081]        3118    24  0.007697  613.889070
9             (618.081, 627.877]        3120    22  0.007051  621.629819
